# 2B DUSt3R Batch Processing 



In [ ]:
from __future__ import annotations

import os, sys, json, math, time, shutil, importlib, traceback
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

try:
    from IPython.display import display
except Exception:
    display = print


## 1. Colab / path setup



In [ ]:
IN_COLAB = 'google.colab' in sys.modules
print('IN_COLAB =', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

COLAB_ROOT = Path('/content') if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path('/content/drive/MyDrive') if IN_COLAB else Path.cwd()
REPO_ROOT = COLAB_ROOT / 'dust3r'

PROJECT_ROOT = DRIVE_ROOT / 'ResearchProject'
DATASET_ROOT = PROJECT_ROOT / 'data/COLMAP_curated'
CARS_ROOT = DATASET_ROOT / 'cars'
METADATA_PATH = DATASET_ROOT / 'curated_metadata.csv'

# New output folder for this batch run
OUTPUT_ROOT = PROJECT_ROOT / 'dust3r_outputs_batch_strict_singlecar_method_v1'
VIS_ROOT = OUTPUT_ROOT / 'visualizations'
RUN_ROOT = OUTPUT_ROOT / 'runs'
SUMMARY_ROOT = OUTPUT_ROOT / 'summary'
INTERACTIVE_ROOT = VIS_ROOT / 'interactive_html'
EXPORT_3DGS_ROOT = PROJECT_ROOT / '3DGSData_dust3r_batch_strict_v1'

for p in [OUTPUT_ROOT, VIS_ROOT, RUN_ROOT, SUMMARY_ROOT, INTERACTIVE_ROOT, EXPORT_3DGS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('CARS_ROOT:', CARS_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('EXPORT_3DGS_ROOT:', EXPORT_3DGS_ROOT)


## 2. Install/import DUSt3R

Run this cell once in Colab. It clones DUSt3R if needed and loads the pretrained model.


In [ ]:
if IN_COLAB and not REPO_ROOT.exists():
    !git clone --recursive https://github.com/naver/dust3r.git /content/dust3r

if IN_COLAB:
    %cd /content/dust3r
    !pip -q install -r requirements.txt
    !pip -q install plotly pandas pillow trimesh huggingface_hub roma
    %cd /content

import torch

assert REPO_ROOT.exists(), f'DUSt3R repo not found: {REPO_ROOT}'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Avoid stale imports after edits/restarts
for name in list(sys.modules.keys()):
    if name == 'dust3r' or name.startswith('dust3r.'):
        del sys.modules[name]
importlib.invalidate_caches()

from dust3r.model import AsymmetricCroCo3DStereo
from dust3r.utils.image import load_images
from dust3r.inference import inference
from dust3r.image_pairs import make_pairs
from dust3r.cloud_opt import global_aligner, GlobalAlignerMode

try:
    import plotly.graph_objects as go
except Exception:
    go = None

MODEL_NAME = 'naver/DUSt3R_ViTLarge_BaseDecoder_512_dpt'
DEVICE_OVERRIDE = None
DEVICE = DEVICE_OVERRIDE or ('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

_t0 = time.time()
model = AsymmetricCroCo3DStereo.from_pretrained(MODEL_NAME).to(DEVICE).eval()
print(f'DUSt3R model loaded in {time.time() - _t0:.1f}s')


## 3. Configuration


In [ ]:
IMAGE_SUBDIR = 'images'
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}

BRUM_CANVAS_W = 512
BRUM_CANVAS_H = 256
BRUM_CANVAS = (BRUM_CANVAS_W, BRUM_CANVAS_H)
BRUM_PREPROCESS_MODE = 'strict_full_frame_resize_512x256'

PAIR_IMAGE_SIZE = 512
GLOBAL_IMAGE_SIZE = 512
GLOBAL_ALIGN_SCHEDULE = 'cosine'
GLOBAL_ALIGN_LR = 0.01
GLOBAL_ALIGN_NITER = 300

BRUM_REAL_CONFIDENCE_GAMMA = 1.5
BRUM_MAX_REAL_H = 0.08

MAX_GLOBAL_CLOUD_PER_VIEW = 15000
MAX_GLOBAL_CLOUD_POINTS = 80000
SAVE_INTERACTIVE_HTML = True
EXPORT_TO_3DGS = True
MAKE_PLOTS = True
FORCE_REPROCESS = False
STOP_ON_ERROR = False

# Batch controls
RUN_ONLY = None       # e.g. ['car_25581194']
MAX_CARS = None       # e.g. 3 for a quick test
RUN_SMOKE_TEST = True
SMOKE_TEST_INDEX = 0

# Usability/evaluation heuristics
MIN_REGISTERED_FRAC_FOR_USABLE = 0.75
MIN_SPARSE_POINTS_FOR_USABLE = 2000
MAX_VIEW_CENTROID_DIST_ZSCORE = 2.5
GLOBAL_ALIGNMENT_LOSS_GOOD_THRESHOLD = 0.05
GLOBAL_ALIGNMENT_LOSS_STRICT_THRESHOLD = 0.10

VIEW_ORDER = [
    'front',
    'front_left',
    'side_left',
    'rear_left',
    'rear',
    'rear_right',
    'side_right',
    'front_right',
]

print('Canvas:', BRUM_CANVAS)
print('Global align iterations:', GLOBAL_ALIGN_NITER)


## 4. General helpers and strict preprocessing



In [ ]:
def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def write_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2)


def read_json(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def normalize_view_name(stem: str) -> str:
    s = stem.lower().strip().replace(' ', '_').replace('-', '_')
    aliases = {
        'frontleft': 'front_left', 'frontright': 'front_right',
        'rearleft': 'rear_left', 'rearright': 'rear_right',
        'sideleft': 'side_left', 'sideright': 'side_right',
        'leftside': 'side_left', 'rightside': 'side_right',
        'front_left_view': 'front_left', 'front_right_view': 'front_right',
        'rear_left_view': 'rear_left', 'rear_right_view': 'rear_right',
        'side_left_view': 'side_left', 'side_right_view': 'side_right',
    }
    return aliases.get(s, s)


def subsample_xyz(xyz, max_points=50000, seed=0):
    xyz = np.asarray(xyz)
    if len(xyz) <= max_points:
        return xyz
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(xyz), max_points, replace=False)
    return xyz[idx]


def read_car_images(car_id: str) -> pd.DataFrame:
    car_dir = CARS_ROOT / car_id
    image_dir = car_dir / IMAGE_SUBDIR
    if not image_dir.exists():
        raise FileNotFoundError(f'Image directory not found: {image_dir}')

    records = []
    for p in sorted(image_dir.iterdir()):
        if p.suffix.lower() not in VALID_EXTS:
            continue
        view = normalize_view_name(p.stem)
        mask_candidates = [
            car_dir / 'masks_brum' / f'{p.stem}.png',
            image_dir.parent / 'masks_brum' / f'{p.stem}.png',
        ]
        mask_path = next((m for m in mask_candidates if m.exists()), None)
        with Image.open(p) as im:
            w, h = im.size
        records.append({
            'car_id': car_id,
            'filename': p.name,
            'path': str(p),
            'mask_path': str(mask_path) if mask_path else None,
            'view': view,
            'exists_in_view_order': view in VIEW_ORDER,
            'raw_w': int(w),
            'raw_h': int(h),
        })

    df = pd.DataFrame(records)
    if df.empty:
        raise ValueError(f'No valid images found in {image_dir}')
    df['view_index'] = df['view'].apply(lambda v: VIEW_ORDER.index(v) if v in VIEW_ORDER else -1)
    return df.sort_values(['view_index', 'filename']).reset_index(drop=True)


def prepare_images_strict_512x256(df_images: pd.DataFrame, prep_dir: Path) -> pd.DataFrame:
    """Single-car-method preprocessing: full-frame direct resize to 512x256.

    Masks are resized and saved if available, but DUSt3R itself uses the RGB images.
    """
    ensure_dir(prep_dir)
    for p in prep_dir.glob('*'):
        if p.is_file():
            p.unlink()

    records = []
    for _, row in df_images.iterrows():
        src = Path(row['path'])
        with Image.open(src) as im:
            if im.mode in ('RGBA', 'LA'):
                im = im.convert('RGBA')
                bg = Image.new('RGBA', im.size, (255, 255, 255, 255))
                im = Image.alpha_composite(bg, im).convert('RGB')
            else:
                im = im.convert('RGB')
            img_prepared = im.resize(BRUM_CANVAS, Image.LANCZOS)

        dst = (prep_dir / src.name).with_suffix('.png')
        img_prepared.save(dst)

        prepared_mask_path = None
        if row.get('mask_path') and isinstance(row.get('mask_path'), str) and Path(row['mask_path']).exists():
            with Image.open(row['mask_path']) as m:
                m = m.convert('L').resize(BRUM_CANVAS, Image.NEAREST)
            mdst = prep_dir / f'{src.stem}_mask.png'
            m.save(mdst)
            prepared_mask_path = str(mdst)

        records.append({
            **row.to_dict(),
            'prepared_path': str(dst),
            'prepared_mask_path': prepared_mask_path,
            'prep_w': BRUM_CANVAS_W,
            'prep_h': BRUM_CANVAS_H,
            'preprocess_mode': BRUM_PREPROCESS_MODE,
            'scale_x': BRUM_CANVAS_W / float(row['raw_w']),
            'scale_y': BRUM_CANVAS_H / float(row['raw_h']),
            'letterbox_offset_x': 0,
            'letterbox_offset_y': 0,
        })

    return pd.DataFrame(records).sort_values(['view_index', 'filename']).reset_index(drop=True)


## 5. Pair diagnostics and DUSt3R global alignment



In [ ]:
@dataclass
class PairConfig:
    name: str
    description: str
    min_confidence: float
    keep_top_k: Optional[int]


PAIR_CFG = PairConfig(
    name='closed_adjacent_ring',
    description='Closed circular adjacent viewpoint graph.',
    min_confidence=0.30,
    keep_top_k=None,
)


def build_adjacent_pairs(df: pd.DataFrame, cfg: PairConfig) -> pd.DataFrame:
    dfv = df[df['view'].isin(VIEW_ORDER)].copy().sort_values('view_index').reset_index(drop=True)
    view_to_row = {row['view']: row for _, row in dfv.iterrows()}
    present = set(dfv['view'])
    pairs = []
    for idx, view in enumerate(VIEW_ORDER):
        nxt = VIEW_ORDER[(idx + 1) % len(VIEW_ORDER)]
        if view in present and nxt in present:
            a, b = view_to_row[view], view_to_row[nxt]
            pairs.append({
                'img1': a['filename'], 'img2': b['filename'],
                'path1': a['prepared_path'], 'path2': b['prepared_path'],
                'view1': a['view'], 'view2': b['view'],
                'type': 'closed_adjacent', 'pair_config': cfg.name,
                'min_confidence': cfg.min_confidence, 'keep_top_k': cfg.keep_top_k,
            })
    cols = ['img1','img2','path1','path2','view1','view2','type','pair_config','min_confidence','keep_top_k']
    return pd.DataFrame(pairs, columns=cols)


def unwrap_first_tensor(x):
    if torch.is_tensor(x):
        return x
    if isinstance(x, (list, tuple)):
        for item in x:
            t = unwrap_first_tensor(item)
            if t is not None:
                return t
    return None


def safe_tensor_mean(x):
    t = unwrap_first_tensor(x)
    if t is None:
        return np.nan
    return float(t.detach().float().cpu().mean().item())


def run_dust3r_for_pairs(df_pairs, model, device='cpu', image_size=512):
    results = []
    for _, row in df_pairs.iterrows():
        pair_id = f"{Path(row['img1']).stem}__{Path(row['img2']).stem}"
        print(f"Pair diagnostic: {row['view1']} ↔ {row['view2']}")
        try:
            images = load_images([row['path1'], row['path2']], size=image_size)
            pairs = make_pairs(images, scene_graph='complete', prefilter=None, symmetrize=True)
            output = inference(pairs, model, device=device, batch_size=1, verbose=False)
            conf1 = safe_tensor_mean(output.get('pred1', {}).get('conf', None))
            conf2 = safe_tensor_mean(output.get('pred2', {}).get('conf', None))
            mean_conf = float(np.nanmean([conf1, conf2]))
            results.append({
                'pair_id': pair_id,
                'img1': row['img1'], 'img2': row['img2'],
                'view1': row['view1'], 'view2': row['view2'],
                'status': 'OK',
                'mean_confidence_1': conf1,
                'mean_confidence_2': conf2,
                'mean_confidence': mean_conf,
                'error': '',
            })
        except Exception as e:
            results.append({
                'pair_id': pair_id,
                'img1': row['img1'], 'img2': row['img2'],
                'view1': row['view1'], 'view2': row['view2'],
                'status': 'FAILED',
                'mean_confidence_1': np.nan,
                'mean_confidence_2': np.nan,
                'mean_confidence': np.nan,
                'error': repr(e),
            })
    return pd.DataFrame(results)


def run_dust3r_global_alignment(df_prepared, model, device='cpu', image_size=512, schedule='cosine', lr=0.01, niter=300):
    image_paths = df_prepared['prepared_path'].tolist()
    views = df_prepared['view'].tolist()

    images = load_images(image_paths, size=image_size)
    pairs = make_pairs(images, scene_graph='complete', prefilter=None, symmetrize=True)
    output = inference(pairs, model, device=device, batch_size=1, verbose=False)

    scene = global_aligner(output, device=device, mode=GlobalAlignerMode.PointCloudOptimizer)
    loss = scene.compute_global_alignment(init='mst', niter=niter, schedule=schedule, lr=lr)

    return {
        'scene': scene,
        'views': views,
        'imgs': scene.imgs,
        'focals': scene.get_focals(),
        'poses': scene.get_im_poses(),
        'pts3d': scene.get_pts3d(),
        'masks': scene.get_masks(),
        'confs': scene.get_conf() if hasattr(scene, 'get_conf') else scene.get_confs() if hasattr(scene, 'get_confs') else None,
        'loss': float(loss) if np.isscalar(loss) else float(loss.detach().cpu().item()) if torch.is_tensor(loss) else loss,
    }


def build_global_cloud_from_scene(ga, max_points_per_view=10000):
    clouds = []
    for i, _ in enumerate(ga['views']):
        pts = ga['pts3d'][i].detach().cpu().numpy()
        mask = ga['masks'][i].detach().cpu().numpy().astype(bool)
        xyz = subsample_xyz(pts[mask], max_points=max_points_per_view, seed=i)
        if len(xyz):
            clouds.append(xyz)
    return np.vstack(clouds) if clouds else np.zeros((0, 3), dtype=np.float32)


## 6. Visualization helpers

Camera centers are visualized from the DUSt3R global-alignment camera-to-world poses. 

In [ ]:
def save_fig(fig, path: Path, dpi: int = 200):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=dpi, bbox_inches='tight', pad_inches=0.02)
    plt.close(fig)


def show_images_grid(df: pd.DataFrame, ncols: int = 4, figsize=(14, 7), save_path: Optional[Path] = None):
    if df.empty:
        return
    n = len(df)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis('off')
    for ax, (_, row) in zip(axes, df.iterrows()):
        img_path = row.get('prepared_path', row.get('path'))
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img)
        ax.set_title(f"{row['view']}
{Path(img_path).name}
{img.size[0]}x{img.size[1]}", fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    save_fig(fig, save_path) if save_path is not None else plt.show()


def plot_pair_graph(df_pairs: pd.DataFrame, present_views: List[str], title: str, save_path: Optional[Path] = None):
    if df_pairs.empty:
        return
    coords = {}
    n = len(VIEW_ORDER)
    for i, v in enumerate(VIEW_ORDER):
        theta = 2 * np.pi * i / n
        coords[v] = (np.cos(theta), np.sin(theta))
    fig = plt.figure(figsize=(5, 5))
    for v, (x, y) in coords.items():
        if v in set(present_views):
            plt.scatter(x, y, s=45)
            plt.text(x, y + 0.08, v, ha='center', fontsize=8)
    for _, row in df_pairs.iterrows():
        if row['view1'] in coords and row['view2'] in coords:
            x1, y1 = coords[row['view1']]
            x2, y2 = coords[row['view2']]
            plt.plot([x1, x2], [y1, y2], linewidth=0.8)
    plt.title(title, fontsize=10)
    plt.axis('equal')
    plt.axis('off')
    save_fig(fig, save_path) if save_path is not None else plt.show()


def camera_center_from_cam2world(T_c2w: np.ndarray) -> np.ndarray:
    T_c2w = np.asarray(T_c2w)
    return T_c2w[:3, 3].copy()


def get_pose_np(P):
    return P.detach().cpu().numpy() if torch.is_tensor(P) else np.asarray(P)


def scene_centroid(global_xyz):
    xyz = np.asarray(global_xyz)
    if len(xyz) == 0:
        return np.zeros(3)
    return np.median(xyz, axis=0)


def arrow_end_towards_scene(camera_center, centroid, ray_len):
    direction = centroid - camera_center
    n = np.linalg.norm(direction)
    if n < 1e-12:
        direction = np.array([0, 0, 1.0])
    else:
        direction = direction / n
    return camera_center + ray_len * direction, direction


def set_nice_3d_view(ax, xyz, elev=18, azim=-55):
    xyz = np.asarray(xyz)
    if len(xyz) == 0:
        return
    lo = np.percentile(xyz, 1, axis=0)
    hi = np.percentile(xyz, 99, axis=0)
    center = (lo + hi) / 2
    radius = float(np.max(hi - lo) / 2)
    radius = max(radius, 1e-6)
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)
    ax.set_box_aspect([1, 1, 1])
    ax.view_init(elev=elev, azim=azim)
    ax.set_axis_off()


def plot_pointcloud_3d(xyz, title='Point cloud', max_points=50000, save_path: Optional[Path] = None):
    if len(xyz) == 0:
        return
    xyz_plot = subsample_xyz(xyz, max_points=max_points, seed=0)
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(xyz_plot[:,0], xyz_plot[:,1], xyz_plot[:,2], s=0.08, alpha=0.65, linewidths=0, depthshade=False)
    ax.set_title(title, fontsize=10)
    set_nice_3d_view(ax, xyz_plot)
    save_fig(fig, save_path) if save_path is not None else plt.show()


def plot_multiview_scene(ga, global_xyz, title='DUSt3R multiview global alignment', save_path: Optional[Path] = None, max_points=50000):
    if len(global_xyz) == 0:
        return
    xyz_plot = subsample_xyz(global_xyz, max_points=max_points, seed=0)
    centroid = scene_centroid(xyz_plot)
    extent = np.max(np.percentile(xyz_plot, 99, axis=0) - np.percentile(xyz_plot, 1, axis=0))
    ray_len = max(0.04 * float(extent), 1e-6)

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(xyz_plot[:,0], xyz_plot[:,1], xyz_plot[:,2], s=0.08, alpha=0.55, linewidths=0, depthshade=False)

    centers = []
    for i, view in enumerate(ga['views']):
        T = get_pose_np(ga['poses'][i])
        c = camera_center_from_cam2world(T)
        p2, _ = arrow_end_towards_scene(c, centroid, ray_len)
        centers.append(c)
        ax.scatter(c[0], c[1], c[2], s=14)
        ax.text(c[0], c[1], c[2], str(view), fontsize=6)
        ax.plot([c[0], p2[0]], [c[1], p2[1]], [c[2], p2[2]], linewidth=0.55)

    if centers:
        centers = np.asarray(centers)
        ax.plot(centers[:,0], centers[:,1], centers[:,2], linewidth=0.55)

    ax.set_title(title, fontsize=10)
    set_nice_3d_view(ax, xyz_plot)
    save_fig(fig, save_path) if save_path is not None else plt.show()


def save_interactive_multiview_html(ga, global_xyz, path: Path, title='DUSt3R multiview global alignment', max_points=80000):
    if go is None or len(global_xyz) == 0:
        return None
    xyz_plot = subsample_xyz(global_xyz, max_points=max_points, seed=0)
    centroid = scene_centroid(xyz_plot)
    extent = np.max(np.percentile(xyz_plot, 99, axis=0) - np.percentile(xyz_plot, 1, axis=0))
    ray_len = max(0.045 * float(extent), 1e-6)

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=xyz_plot[:,0], y=xyz_plot[:,1], z=xyz_plot[:,2],
        mode='markers', marker=dict(size=1, opacity=0.45), name='DUSt3R points'
    ))

    centers, labels = [], []
    cone_x=[]; cone_y=[]; cone_z=[]; cone_u=[]; cone_v=[]; cone_w=[]
    for i, view in enumerate(ga['views']):
        T = get_pose_np(ga['poses'][i])
        c = camera_center_from_cam2world(T)
        p2, direction = arrow_end_towards_scene(c, centroid, ray_len)
        centers.append(c); labels.append(str(view))

        fig.add_trace(go.Scatter3d(
            x=[c[0], p2[0]], y=[c[1], p2[1]], z=[c[2], p2[2]],
            mode='lines', line=dict(width=2), name=f'{view} → scene', showlegend=False
        ))
        cone_x.append(p2[0]); cone_y.append(p2[1]); cone_z.append(p2[2])
        cone_u.append(direction[0]); cone_v.append(direction[1]); cone_w.append(direction[2])

    if centers:
        centers = np.asarray(centers)
        fig.add_trace(go.Scatter3d(
            x=centers[:,0], y=centers[:,1], z=centers[:,2],
            mode='markers+lines+text', text=labels, textposition='top center',
            marker=dict(size=3), line=dict(width=1), name='camera centers'
        ))
        # Small cone tips make the thin camera rays readable as arrows.
        fig.add_trace(go.Cone(
            x=cone_x, y=cone_y, z=cone_z,
            u=cone_u, v=cone_v, w=cone_w,
            sizemode='absolute', sizeref=ray_len * 0.12,
            anchor='tip', showscale=False, name='arrow tips', opacity=0.75
        ))

    fig.update_layout(title=title, scene=dict(aspectmode='data'), height=800)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(str(path), include_plotlyjs='cdn')
    return fig


## 7. Evaluation/statistics helpers and COLMAP export


In [ ]:
def invert_pose(T: np.ndarray) -> np.ndarray:
    T = np.asarray(T)
    R = T[:3, :3]
    t = T[:3, 3]
    T_inv = np.eye(4, dtype=T.dtype)
    T_inv[:3, :3] = R.T
    T_inv[:3, 3] = -R.T @ t
    return T_inv


def rotmat_to_qvec(R: np.ndarray) -> np.ndarray:
    K = np.array([
        [R[0, 0] - R[1, 1] - R[2, 2], 0, 0, 0],
        [R[1, 0] + R[0, 1], R[1, 1] - R[0, 0] - R[2, 2], 0, 0],
        [R[2, 0] + R[0, 2], R[2, 1] + R[1, 2], R[2, 2] - R[0, 0] - R[1, 1], 0],
        [R[1, 2] - R[2, 1], R[2, 0] - R[0, 2], R[0, 1] - R[1, 0], R[0, 0] + R[1, 1] + R[2, 2]],
    ]) / 3.0
    eigvals, eigvecs = np.linalg.eigh(K)
    qvec = eigvecs[:, np.argmax(eigvals)]
    qvec = np.array([qvec[3], qvec[0], qvec[1], qvec[2]])
    if qvec[0] < 0:
        qvec *= -1
    return qvec


def world2cam_from_pose_tensor(T_pose: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    T_w2c = invert_pose(T_pose)
    return T_w2c[:3, :3], T_w2c[:3, 3]


def shared_K(width=BRUM_CANVAS_W, height=BRUM_CANVAS_H):
    # Fallback approximate intrinsics for COLMAP text export. 3A/4B can overwrite if needed.
    f = float(max(width, height))
    return np.array([[f, 0, width/2], [0, f, height/2], [0, 0, 1.0]], dtype=np.float64)


def write_colmap_text_model(export_dir: Path, df_car: pd.DataFrame, ga: dict, sparse_xyz: np.ndarray):
    sparse_dir = ensure_dir(export_dir / 'sparse' / '0')
    images_dir = ensure_dir(export_dir / 'images')

    df_car_local = df_car.copy().reset_index(drop=True)
    for _, row in df_car_local.iterrows():
        shutil.copy2(row['prepared_path'], images_dir / Path(row['prepared_path']).name)

    K = shared_K()
    camera_id = 1
    camera_lines = [
        '# Camera list with one line of data per camera:
',
        '# CAMERA_ID, MODEL, WIDTH, HEIGHT, PARAMS[]
',
        f'{camera_id} PINHOLE {BRUM_CANVAS_W} {BRUM_CANVAS_H} {K[0,0]:.12f} {K[1,1]:.12f} {K[0,2]:.12f} {K[1,2]:.12f}
',
    ]

    image_lines = ['# Image list with two lines of data per image:
']
    for i, row in df_car_local.iterrows():
        T = get_pose_np(ga['poses'][i])
        R, t = world2cam_from_pose_tensor(T)
        q = rotmat_to_qvec(R)
        img_name = Path(row['prepared_path']).name
        image_lines.append(
            f"{i+1} {q[0]:.12f} {q[1]:.12f} {q[2]:.12f} {q[3]:.12f} "
            f"{t[0]:.12f} {t[1]:.12f} {t[2]:.12f} {camera_id} {img_name}

"
        )

    sparse_xyz = np.asarray(sparse_xyz)
    point_lines = ['# 3D point list:
']
    for pid, p in enumerate(sparse_xyz, start=1):
        point_lines.append(f'{pid} {p[0]:.8f} {p[1]:.8f} {p[2]:.8f} 128 128 128 1.0
')

    (sparse_dir / 'cameras.txt').write_text(''.join(camera_lines), encoding='utf-8')
    (sparse_dir / 'images.txt').write_text(''.join(image_lines), encoding='utf-8')
    (sparse_dir / 'points3D.txt').write_text(''.join(point_lines), encoding='utf-8')

    return {'images_dir': str(images_dir), 'sparse_dir': str(sparse_dir), 'n_images': len(df_car_local), 'n_points3D': len(sparse_xyz)}


def flag_bad_views_after_global_alignment(ga):
    rows = []
    centroids = []
    for i, view in enumerate(ga['views']):
        pts = ga['pts3d'][i].detach().cpu().numpy()
        mask = ga['masks'][i].detach().cpu().numpy().astype(bool)
        valid = pts[mask]
        if len(valid):
            c = np.median(valid, axis=0)
        else:
            c = np.array([np.nan, np.nan, np.nan])
        centroids.append(c)
        rows.append({'view': view, 'n_valid_points': int(len(valid)), 'centroid_x': c[0], 'centroid_y': c[1], 'centroid_z': c[2]})
    df = pd.DataFrame(rows)
    C = np.asarray(centroids, dtype=float)
    if np.isfinite(C).all() and len(C) > 1:
        global_c = np.nanmedian(C, axis=0)
        d = np.linalg.norm(C - global_c, axis=1)
        z = (d - np.nanmean(d)) / (np.nanstd(d) + 1e-9)
        df['centroid_dist'] = d
        df['centroid_dist_zscore'] = z
        df['flagged_bad_view'] = z > MAX_VIEW_CENTROID_DIST_ZSCORE
    else:
        df['centroid_dist'] = np.nan
        df['centroid_dist_zscore'] = np.nan
        df['flagged_bad_view'] = False
    return df


def summarize_success(car_id, df_images, df_pairs, pair_results, ga, global_xyz, df_bad_views, elapsed_sec):
    n_pairs = len(df_pairs)
    n_ok_pairs = int(pair_results['status'].astype(str).str.upper().eq('OK').sum()) if len(pair_results) else 0
    pair_success_rate = float(n_ok_pairs / n_pairs) if n_pairs else 0.0

    if ga is not None:
        per_view_npts = [int(m.detach().cpu().numpy().astype(bool).sum()) for m in ga['masks']]
        n_registered = int(sum(n > 0 for n in per_view_npts))
        registered_frac = float(n_registered / len(ga['views'])) if len(ga['views']) else 0.0
        loss = float(ga['loss']) if ga.get('loss', None) is not None else np.nan
    else:
        per_view_npts = []
        n_registered = 0
        registered_frac = 0.0
        loss = np.nan

    n_flagged = int(df_bad_views['flagged_bad_view'].sum()) if df_bad_views is not None and len(df_bad_views) else 0
    n_sparse = int(len(global_xyz))
    usable_relaxed = bool((ga is not None) and registered_frac >= MIN_REGISTERED_FRAC_FOR_USABLE and n_sparse >= MIN_SPARSE_POINTS_FOR_USABLE and n_flagged == 0)
    usable_strict = bool(usable_relaxed and (np.isnan(loss) or loss <= GLOBAL_ALIGNMENT_LOSS_STRICT_THRESHOLD))
    needs_review = bool((ga is None) or registered_frac < 1.0 or n_flagged > 0 or (pd.notna(loss) and loss > GLOBAL_ALIGNMENT_LOSS_GOOD_THRESHOLD))

    if ga is None:
        label = 'failed'
    elif not usable_relaxed:
        label = 'reject'
    elif usable_strict and not needs_review:
        label = 'good_candidate'
    elif usable_strict:
        label = 'strict_candidate_review'
    else:
        label = 'relaxed_candidate_review'

    return {
        'car_id': car_id,
        'elapsed_sec': float(elapsed_sec),
        'n_input_images': int(len(df_images)),
        'n_expected_view_images': int(df_images['view'].isin(VIEW_ORDER).sum()),
        'n_pairs_closed_ring': int(n_pairs),
        'n_ok_pairs': int(n_ok_pairs),
        'pair_success_rate': pair_success_rate,
        'global_alignment_ok': ga is not None,
        'global_alignment_loss': loss,
        'n_registered_images': int(n_registered),
        'registered_frac': registered_frac,
        'n_sparse_points_multiview_ga': n_sparse,
        'per_view_valid_points': per_view_npts,
        'n_flagged_bad_views_ga': n_flagged,
        'usable_strict_candidate': usable_strict,
        'usable_relaxed_candidate': usable_relaxed,
        'needs_visual_review': needs_review,
        'ga_quality_label': label,
        'preprocess_mode': BRUM_PREPROCESS_MODE,
        'canvas_w': BRUM_CANVAS_W,
        'canvas_h': BRUM_CANVAS_H,
        'note': 'Global alignment poses are source of truth; pairwise diagnostics are not used as final poses.',
    }


## 8. Per-scene processing function


In [ ]:
def to_cpu_detached(obj):
    if torch.is_tensor(obj):
        return obj.detach().cpu()
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            # The raw scene object is not safely serializable; keep the tensors/metadata only.
            if k == 'scene':
                continue
            out[k] = to_cpu_detached(v)
        return out
    if isinstance(obj, list):
        return [to_cpu_detached(v) for v in obj]
    if isinstance(obj, tuple):
        return tuple(to_cpu_detached(v) for v in obj)
    return obj


def process_single_car_batch(car_id: str, export_to_3dgs=True, make_plots=True, force_reprocess=False):
    t0 = time.time()
    car_out = ensure_dir(OUTPUT_ROOT / car_id)
    car_run = ensure_dir(RUN_ROOT / car_id)
    car_vis = ensure_dir(VIS_ROOT / car_id)
    car_html = ensure_dir(INTERACTIVE_ROOT / car_id)
    prep_dir = ensure_dir(car_out / 'prepared_images')

    summary_json = car_out / 'summary.json'
    if summary_json.exists() and not force_reprocess:
        summary = read_json(summary_json)
        summary['reused_from_cache'] = True
        return {'summary': summary, 'reused': True}

    df_images = read_car_images(car_id)
    df_prepared = prepare_images_strict_512x256(df_images, prep_dir)
    df_prepared.to_csv(car_out / 'prepared_camera_manifest.csv', index=False)
    write_json({'width': BRUM_CANVAS_W, 'height': BRUM_CANVAS_H, 'preprocess_mode': BRUM_PREPROCESS_MODE}, car_out / 'brum_camera_model.json')

    df_pairs = build_adjacent_pairs(df_prepared, PAIR_CFG)
    ensure_dir(car_out / 'pair_lists')
    df_pairs.to_csv(car_out / 'pair_lists' / f'{PAIR_CFG.name}.csv', index=False)

    if make_plots:
        show_images_grid(df_prepared, save_path=car_vis / 'input_images_grid.png')
        plot_pair_graph(df_pairs, df_prepared['view'].tolist(), f'{car_id}: closed adjacent pair graph', save_path=car_vis / 'pair_graph_closed_ring.png')

    pair_results = run_dust3r_for_pairs(df_pairs, model, device=DEVICE, image_size=PAIR_IMAGE_SIZE)
    pair_results.to_csv(car_run / 'pair_run_results.csv', index=False)

    ga = run_dust3r_global_alignment(
        df_prepared, model, device=DEVICE, image_size=GLOBAL_IMAGE_SIZE,
        schedule=GLOBAL_ALIGN_SCHEDULE, lr=GLOBAL_ALIGN_LR, niter=GLOBAL_ALIGN_NITER,
    )
    ga_cpu = to_cpu_detached(ga)
    torch.save(ga_cpu, car_out / 'global_alignment.pt')

    global_xyz = build_global_cloud_from_scene(ga, max_points_per_view=MAX_GLOBAL_CLOUD_PER_VIEW)
    np.save(car_out / 'global_xyz_multiview.npy', global_xyz)

    df_bad = flag_bad_views_after_global_alignment(ga)
    df_bad.to_csv(car_run / 'global_alignment_view_diagnostics.csv', index=False)

    if make_plots:
        plot_pointcloud_3d(global_xyz, title=f'{car_id}: DUSt3R multiview sparse cloud', save_path=car_vis / 'multiview_sparse_cloud.png')
        plot_multiview_scene(ga, global_xyz, title=f'{car_id}: DUSt3R global alignment', save_path=car_vis / 'multiview_global_alignment.png')
        if SAVE_INTERACTIVE_HTML:
            save_interactive_multiview_html(ga, global_xyz, car_html / 'multiview_global_alignment.html', title=f'{car_id}: DUSt3R global alignment')

    summary = summarize_success(car_id, df_images, df_pairs, pair_results, ga, global_xyz, df_bad, time.time() - t0)

    if export_to_3dgs and ga is not None and len(global_xyz) > 0:
        export_dir = EXPORT_3DGS_ROOT / car_id
        export_info = write_colmap_text_model(export_dir, df_prepared, ga, global_xyz)
        summary['exported_to_3dgs'] = True
        summary['export_dir'] = str(export_dir)
        summary['export_info'] = export_info
    else:
        summary['exported_to_3dgs'] = False
        summary['export_dir'] = None

    summary['html_path'] = str(car_html / 'multiview_global_alignment.html')
    summary['visualization_dir'] = str(car_vis)
    summary['reused_from_cache'] = False
    write_json(summary, summary_json)

    return {
        'summary': summary,
        'df_images': df_images,
        'df_prepared': df_prepared,
        'df_pairs': df_pairs,
        'pair_results': pair_results,
        'ga': ga,
        'global_xyz': global_xyz,
        'df_bad_views': df_bad,
        'reused': False,
    }


## 9. Discover available cars and run optional smoke test


In [ ]:
if METADATA_PATH.exists():
    df_meta = pd.read_csv(METADATA_PATH)
    display(df_meta.head())
else:
    df_meta = pd.DataFrame()
    print('Metadata CSV not found:', METADATA_PATH)

available_car_dirs = sorted([p.name for p in CARS_ROOT.iterdir() if p.is_dir()]) if CARS_ROOT.exists() else []
print('Found cars:', len(available_car_dirs))
print(available_car_dirs)

if RUN_SMOKE_TEST and available_car_dirs:
    smoke_id = available_car_dirs[min(SMOKE_TEST_INDEX, len(available_car_dirs)-1)]
    print('Smoke test car:', smoke_id)
    smoke = process_single_car_batch(smoke_id, export_to_3dgs=EXPORT_TO_3DGS, make_plots=True, force_reprocess=FORCE_REPROCESS)
    display(pd.DataFrame([smoke['summary']]).T)
else:
    print('Smoke test skipped')


## 10. Run full batch with incremental saving

Set `RUN_ONLY` or `MAX_CARS` above for smaller runs.


In [ ]:
car_ids = available_car_dirs if RUN_ONLY is None else [c for c in available_car_dirs if c in set(RUN_ONLY)]
if MAX_CARS is not None:
    car_ids = car_ids[:MAX_CARS]

print('Cars to process:', len(car_ids))
summary_rows = []
error_rows = []

summary_csv = SUMMARY_ROOT / 'dust3r_batch_strict_summary.csv'
errors_csv = SUMMARY_ROOT / 'dust3r_batch_strict_errors.csv'

for idx, car_id in enumerate(car_ids, start=1):
    print(f'
[{idx}/{len(car_ids)}] {car_id}')
    try:
        result = process_single_car_batch(car_id, export_to_3dgs=EXPORT_TO_3DGS, make_plots=MAKE_PLOTS, force_reprocess=FORCE_REPROCESS)
        summary_rows.append(result['summary'])
    except Exception as e:
        msg = str(e)
        print(f'[FAILED] {car_id}: {msg}')
        traceback.print_exc()
        error_rows.append({'car_id': car_id, 'error': msg, 'traceback': traceback.format_exc()})
        if STOP_ON_ERROR:
            raise

    pd.DataFrame(summary_rows).to_csv(summary_csv, index=False)
    pd.DataFrame(error_rows).to_csv(errors_csv, index=False)
    print('Incremental summary saved:', summary_csv)

summary_df = pd.DataFrame(summary_rows)
errors_df = pd.DataFrame(error_rows)

if not summary_df.empty:
    sort_cols = [c for c in ['usable_strict_candidate', 'registered_frac', 'n_sparse_points_multiview_ga'] if c in summary_df.columns]
    if sort_cols:
        summary_df = summary_df.sort_values(sort_cols, ascending=[False] * len(sort_cols))

summary_df.to_csv(summary_csv, index=False)
errors_df.to_csv(errors_csv, index=False)

print('Saved summary to', summary_csv)
print('Saved errors to', errors_csv)
display(summary_df)
display(errors_df)


## 11. Evaluation tables and compact report statistics


In [ ]:
if 'summary_df' not in globals() or summary_df.empty:
    summary_csv = SUMMARY_ROOT / 'dust3r_batch_strict_summary.csv'
    summary_df = pd.read_csv(summary_csv) if summary_csv.exists() else pd.DataFrame()

if len(summary_df):
    clean_cols = [
        'car_id', 'n_input_images', 'n_expected_view_images',
        'n_pairs_closed_ring', 'n_ok_pairs', 'pair_success_rate',
        'global_alignment_ok', 'global_alignment_loss',
        'n_registered_images', 'registered_frac', 'n_sparse_points_multiview_ga',
        'n_flagged_bad_views_ga', 'usable_strict_candidate', 'usable_relaxed_candidate',
        'needs_visual_review', 'ga_quality_label', 'exported_to_3dgs',
        'visualization_dir', 'html_path', 'export_dir',
    ]
    clean_cols = [c for c in clean_cols if c in summary_df.columns]
    clean_eval = summary_df[clean_cols].copy()
    clean_path = SUMMARY_ROOT / 'dust3r_batch_strict_eval_clean.csv'
    clean_eval.to_csv(clean_path, index=False)

    print('Quality label counts:')
    display(clean_eval['ga_quality_label'].value_counts(dropna=False).to_frame('count'))

    print('Core statistics:')
    stats = {
        'n_scenes': len(clean_eval),
        'global_alignment_ok_rate': float(clean_eval['global_alignment_ok'].mean()) if 'global_alignment_ok' in clean_eval else np.nan,
        'mean_registered_frac': float(clean_eval['registered_frac'].mean()) if 'registered_frac' in clean_eval else np.nan,
        'median_registered_frac': float(clean_eval['registered_frac'].median()) if 'registered_frac' in clean_eval else np.nan,
        'mean_global_alignment_loss': float(clean_eval['global_alignment_loss'].mean()) if 'global_alignment_loss' in clean_eval else np.nan,
        'median_global_alignment_loss': float(clean_eval['global_alignment_loss'].median()) if 'global_alignment_loss' in clean_eval else np.nan,
        'mean_sparse_points': float(clean_eval['n_sparse_points_multiview_ga'].mean()) if 'n_sparse_points_multiview_ga' in clean_eval else np.nan,
        'strict_candidate_count': int(clean_eval['usable_strict_candidate'].sum()) if 'usable_strict_candidate' in clean_eval else 0,
        'relaxed_candidate_count': int(clean_eval['usable_relaxed_candidate'].sum()) if 'usable_relaxed_candidate' in clean_eval else 0,
    }
    stats_df = pd.DataFrame([stats])
    stats_path = SUMMARY_ROOT / 'dust3r_batch_strict_report_stats.csv'
    stats_df.to_csv(stats_path, index=False)
    display(stats_df.T.rename(columns={0:'value'}))

    print('Clean evaluation saved to:', clean_path)
    print('Report stats saved to:', stats_path)
    display(clean_eval)
else:
    print('No summary available yet.')


## 12. Verify exported 3DGS folders


In [ ]:
exported = []
for car_dir in sorted(EXPORT_3DGS_ROOT.iterdir()) if EXPORT_3DGS_ROOT.exists() else []:
    if not car_dir.is_dir():
        continue
    sparse_dir = car_dir / 'sparse' / '0'
    images_dir = car_dir / 'images'
    exported.append({
        'car_id': car_dir.name,
        'has_images_dir': images_dir.exists(),
        'has_sparse_dir': sparse_dir.exists(),
        'has_cameras_txt': (sparse_dir / 'cameras.txt').exists(),
        'has_images_txt': (sparse_dir / 'images.txt').exists(),
        'has_points3D_txt': (sparse_dir / 'points3D.txt').exists(),
    })
exported_df = pd.DataFrame(exported)
display(exported_df)
exported_df.to_csv(SUMMARY_ROOT / 'exported_3dgs_folder_check.csv', index=False)
